# Constraint Optimization


**约束优化，也可称为“约束规划（Constraint Programming）”**，是指从非常多的候选方案中识别和选择可行解决方案的方法，其中问题可以根据任意约束进行建模。CP问题出现在许多应用工程学科中。

- **它基于可行性，找到可行的解决方案，而不完全是优化**（找到最优解决方案），并且更侧重于约束和变量而不是目标函数。事实上，CP问题甚至可能没有目标函数。使用CP是为了通过向问题添加约束，将一大堆可能的解决方案缩小到更易于管理的子集。

- 常见的约束优化问题包括员工排程、作业车间调度等。还有一些平时比较熟悉的算法问题，比如N皇后、数独求解等也可归纳到这一范畴中。


--------

关于求解器本身不再赘述，Ortools提供了几种解决CP问题的方法：
1. CP-SAT 求解器：使用SAT（满足性）方法的约束规划求解器，这是在原始CP求解器基础上迭代的新版本；
2. 原始的CP求解器。

P.S. 如果可以使用线性目标和线性约束对问题进行建模，那么可以考虑 **MPSolver**。ortools官网同样补充，遇到路径问题等，通常使用专门的车辆路径库来解决（即使它们可以用线性模型表示）。


---------

后面展示的代码是通过CP-SAT方法求约束规划问题的实战部分。由于官网最开始的示例有点简单，所以直接用了官网后面的一个案例：N皇后问题，以及我之前做过的数独求解的案例进行展示。


以N皇后为例，为了统计 $N \times N$ 的棋盘上放 N 个皇后，使得其同行同列同对角线不冲突，总共有多少种放法。我们要做的就是输出 $N \times N$ 的棋盘，使得“每一列都不同、每一行都不同、对角线也重复”。借助`AddAllDifferent` 方法，可以在遍历的过程中批量地加入约束，减少一些不必要的 `AddConstraint` 操作。

另一个案例是数独求解，我们需要输入一个 $9 \times 9$ 的棋盘，规定其中一些值为常量，剩下的值在1～9之间选取，选取的结果要满足“行、列、宫”约束。

> 在这种思路下，我们可以求解所有的变形数独问题。譬如，杀手数独可以通过增加“小宫内的数字和为某个定值”来实现，而“锯齿数独”则可以通过改变宫的编号，实现约束的灵活变化。



## N-Queens

In [7]:
"""OR-Tools solution to the N-queens problem."""
import time
from ortools.sat.python import cp_model
class NQueenSolutionPrinter(cp_model.CpSolverSolutionCallback):
    """Print intermediate solutions."""

    def __init__(self, queens):
        cp_model.CpSolverSolutionCallback.__init__(self)
        self.__queens = queens
        self.__solution_count = 0
        self.__start_time = time.time()

    def solution_count(self):
        return self.__solution_count

    def on_solution_callback(self):
        current_time = time.time()
        print(
            f"Solution {self.__solution_count}, "
            f"time = {current_time - self.__start_time} s"
        )
        self.__solution_count += 1

        all_queens = range(len(self.__queens))
        for i in all_queens:
            for j in all_queens:
                if self.Value(self.__queens[j]) == i:
                    # There is a queen in column j, row i.
                    print("Q", end=" ")
                else:
                    print("_", end=" ")
            print()
        print()


def main(board_size):
    model = cp_model.CpModel() # 创建求解器

    # 创建变量，变量的下标就是其所处的列，变量的值就是其所在的行
    queens = [model.NewIntVar(0, \
        board_size - 1, f"x_{i}") for i in range(board_size)]

    # 约束条件
    model.AddAllDifferent(queens)

    # 对角线不能有冲突
    model.AddAllDifferent(queens[i] + i for i in range(board_size))
    model.AddAllDifferent(queens[i] - i for i in range(board_size))

    # 模型求解
    solver = cp_model.CpSolver()
    solution_printer = NQueenSolutionPrinter(queens)
    solver.parameters.enumerate_all_solutions = True
    solver.Solve(model, solution_printer)

    print("\nStatistics")
    print(f"  conflicts      : {solver.NumConflicts()}")
    print(f"  branches       : {solver.NumBranches()}")
    print(f"  wall time      : {solver.WallTime()} s")
    print(f"  solutions found: {solution_printer.solution_count()}")

if __name__ == "__main__":
    # 这里求解一个 11 皇后问题
    size = 11
    main(size)

Solution 0, time = 0.009324073791503906 s
_ _ _ _ _ Q _ _ _ _ _ 
Q _ _ _ _ _ _ _ _ _ _ 
_ _ _ _ _ _ Q _ _ _ _ 
_ Q _ _ _ _ _ _ _ _ _ 
_ _ _ _ _ _ _ Q _ _ _ 
_ _ Q _ _ _ _ _ _ _ _ 
_ _ _ _ _ _ _ _ Q _ _ 
_ _ _ Q _ _ _ _ _ _ _ 
_ _ _ _ _ _ _ _ _ Q _ 
_ _ _ _ Q _ _ _ _ _ _ 
_ _ _ _ _ _ _ _ _ _ Q 

Solution 1, time = 0.009979009628295898 s
_ _ _ _ _ Q _ _ _ _ _ 
Q _ _ _ _ _ _ _ _ _ _ 
_ _ _ _ _ _ Q _ _ _ _ 
_ _ _ _ _ _ _ _ _ Q _ 
_ _ _ _ _ _ _ Q _ _ _ 
_ _ Q _ _ _ _ _ _ _ _ 
_ _ _ _ _ _ _ _ Q _ _ 
_ _ _ Q _ _ _ _ _ _ _ 
_ Q _ _ _ _ _ _ _ _ _ 
_ _ _ _ Q _ _ _ _ _ _ 
_ _ _ _ _ _ _ _ _ _ Q 

Solution 2, time = 0.010807991027832031 s
_ _ _ _ _ Q _ _ _ _ _ 
_ _ _ _ _ _ _ _ Q _ _ 
_ _ _ _ _ _ Q _ _ _ _ 
_ Q _ _ _ _ _ _ _ _ _ 
_ _ _ _ _ _ _ Q _ _ _ 
_ _ Q _ _ _ _ _ _ _ _ 
Q _ _ _ _ _ _ _ _ _ _ 
_ _ _ Q _ _ _ _ _ _ _ 
_ _ _ _ _ _ _ _ _ Q _ 
_ _ _ _ Q _ _ _ _ _ _ 
_ _ _ _ _ _ _ _ _ _ Q 

Solution 3, time = 0.011275053024291992 s
_ _ _ _ _ Q _ _ _ _ _ 
_ _ _ _ _ _ _ _ Q _ _ 
_ Q _ _ _ _ _ _ _ _ _ 
_

In [ ]:
from ortools.linear_solver import pywraplp
import math

def iterative_solution_with_lazy_constraints():
    """迭代求解：模拟惰性约束添加"""
    iteration = 0
    best_solution = None
    best_objective = -float('inf')
    
    while iteration < 10:  # 最多迭代10次
        iteration += 1
        print(f"\n=== 第 {iteration} 次迭代 ===")
        
        # 创建新求解器实例
        solver = pywraplp.Solver.CreateSolver('CBC')
        
        # 定义变量
        x = []
        n = 20
        for i in range(n):
            x.append(solver.IntVar(0, 1, f'x{i}'))
        
        # 基础约束
        solver.Add(solver.Sum(x) <= 10)
        
        # 如果有之前的最优解，添加割平面（排除当前解）
        if best_solution is not None:
            # 添加一个简单的割平面：排除当前解
            # 这只是一个示例，实际应用中应根据问题特点设计割平面
            constraint = solver.Sum([x[i] for i in range(n) 
                                   if best_solution[i] > 0.5])
            solver.Add(constraint <= 9)  # 强制改变至少一个变量的取值
        
        # 目标函数：最大化权重和
        weights = [i+1 for i in range(n)]  # 简单权重
        objective = solver.Sum([weights[i] * x[i] for i in range(n)])
        solver.Maximize(objective)
        
        # 求解
        result_status = solver.Solve()
        
        if result_status == pywraplp.Solver.OPTIMAL:
            current_obj = solver.Objective().Value()
            current_sol = [var.solution_value() for var in x]
            
            print(f"当前目标值: {current_obj}")
            print(f"当前解: {[int(val) for val in current_sol]}")
            
            # 检查是否需要添加更多约束
            if current_obj > best_objective + 1e-6:
                best_objective = current_obj
                best_solution = current_sol
                print(f"找到改进解，目标值: {best_objective}")
            else:
                print("未找到改进解，停止迭代")
                break
        else:
            print("求解失败")
            break
    
    print(f"\n=== 最终结果 ===")
    print(f"迭代次数: {iteration}")
    print(f"最佳目标值: {best_objective}")
    if best_solution:
        print(f"最佳解: {[int(val) for val in best_solution]}")
    
    return best_objective, best_solution

if __name__ == '__main__':
    iterative_solution_with_lazy_constraints()


=== 第 1 次迭代 ===
当前目标值: 155.0
当前解: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
找到改进解，目标值: 155.0

=== 第 2 次迭代 ===
当前目标值: 154.0
当前解: [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]
未找到改进解，停止迭代

=== 最终结果 ===
迭代次数: 2
最佳目标值: 155.0
最佳解: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [ ]:
z = "- 2 5 13 7 1 3 7 4 2 11\n5 3 1 1 2 1 2 5 1 1 1\n1 0 0 3 2 0 0 1 3 3 5\n5 0 3 5 4 1 1 3 1 2 5\n8 2 2 3 1 3 5 1 2 3 3\n3 4 0 1 5 2 0 3 1 5 2\n6 2 4 2 4 2 1 0 2 1 1\n9 2 3 4 2 5 4 4 2 3 3\n8 2 2 3 3 2 2 3 4 5 1\n3 3 5 3 4 2 5 2 0 1 5\n7 2 3 2 4 3 1 4 2 5 4".split("\n")


- 2 5 13 7 1 3 7 4 2 11


In [ ]:
a = """x - se ew ew ew sw\nse ew nw - se ew nw\nne ew sw x ns - x\nx - ns - ns se sw\nse ew nw x ne nw ns\nns x se ew sw - ns\nne ew nw x ne ew nw""".split("\n")
a = [row.split(" ") for row in a]
for i in range(len(a)):
    for j in range(len(a)):
        if len(a[i][j]) > 1:
            a[i][j] = "".join(sorted(a[i][j]))
"\n".join([" ".join(row) for row in a])


'x - es ew ew ew sw\nes ew nw - es ew nw\nen ew sw x ns - x\nx - ns - ns es sw\nes ew nw x en nw ns\nns x es ew sw - ns\nen ew nw x en ew nw'

In [ ]:
def decode_arrow_number_16(encoded_str, width, height):
    """
    解码 Yajilin 的箭头数字编码
    
    编码格式：
    1. '+' : qdir=0, qnum=-3
    2. '0'-'4' + 数字 : 方向=0-4, 数字=1位16进制
    3. '5'-'9' + 2位数字 : 方向=0-4, 数字=2位16进制
    4. '-' + 方向 + 3位数字 : 方向=0-4, 数字=3位16进制(256-4095)
    5. 'a'-'z' : 跳过 (字符-10) 个单元格
    """
    total_cells = width * height
    cells = [{'qnum': -1, 'qdir': 0} for _ in range(total_cells)]  # 默认无数字
    
    i = 0  # 编码字符串索引
    c = 0  # 单元格索引
    
    while i < len(encoded_str) and c < total_cells:
        ch = encoded_str[i]
        
        if ch == '+':
            # 特殊标记
            cells[c] = {'qnum': -3, 'qdir': 0}
            i += 1
            
        elif '0' <= ch <= '4':
            # 格式: 方向(0-4) + 数字
            direction = int(ch, 16)  # 0-4
            if i + 1 < len(encoded_str):
                next_ch = encoded_str[i + 1]
                if next_ch == '.':
                    number = -2  # 问号
                else:
                    number = int(next_ch, 16)  # 0-15
                cells[c] = {'qnum': number, 'qdir': direction}
                i += 2
            else:
                i += 1
                
        elif '5' <= ch <= '9':
            # 格式: 方向(0-4) + 2位数字
            direction = int(ch, 16) - 5
            if i + 2 < len(encoded_str):
                number = int(encoded_str[i+1:i+3], 16)  # 2位16进制
                cells[c] = {'qnum': number, 'qdir': direction}
                i += 3
            else:
                i += 1
                
        elif ch == '-':
            # 格式: - + 方向 + 3位数字
            if i + 4 < len(encoded_str):
                direction = int(encoded_str[i+1], 16)
                number = int(encoded_str[i+2:i+5], 16)  # 3位16进制
                cells[c] = {'qnum': number, 'qdir': direction}
                i += 5
            else:
                i += 1
                
        elif 'a' <= ch <= 'z':
            # 跳过空白单元格
            skip = int(ch, 36) - 10
            c += skip
            i += 1
        else:
            i += 1
        
        c += 1
    
    return cells
# 40b22e20f22k21b44c42f31a21v12h3321l10h30
res = decode_arrow_number_16("23h2323j252223zzb33g121112f33b3311z11f33p33h41u34a13e131313f32a13a11zf20c30g3232r11f30m12", 17, 17)
print(res)

[{'qnum': 0, 'qdir': 4}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': 2, 'qdir': 2}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': 0, 'qdir': 2}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': 2, 'qdir': 2}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': 1, 'qdir': 2}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': 4, 'qdir': 4}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': 2, 'qdir': 4}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum': -1, 'qdir': 0}, {'qnum

In [ ]:
def convert_encoding(old_value):
    
    left = (old_value >> 0) & 1  # 第0位：左
    bottom = (old_value >> 1) & 1  # 第1位：下
    right = (old_value >> 2) & 1  # 第2位：右
    top = (old_value >> 3) & 1  # 第3位：上
    

    new_value = 0
    new_value |= (top << 0)    # 旧上 -> 新上
    new_value |= (right << 1)  # 旧右 -> 新右
    new_value |= (bottom << 2)  # 旧下 -> 新下
    new_value |= (left << 3)   # 旧左 -> 新左
    
    return new_value


if __name__ == "__main__":
    old_encoding = 11  
    new_encoding = convert_encoding(old_encoding)
    
    print(new_encoding)

13


In [ ]:
import requests
from typing import List, Dict, Any, Optional


class PuzzleKitClient:
    """PuzzleKit API 客户端"""
    
    def __init__(self, base_url: str = "https://puzzlekit-api.onrender.com"):
        self.base_url = base_url.rstrip("/")
        self.timeout = 60  # Render 免费版可能较慢
    
    def health_check(self) -> Dict[str, Any]:
        """检查 API 状态"""
        response = requests.get(f"{self.base_url}/", timeout=self.timeout)
        return response.json()
    
    def list_solvers(self) -> List[str]:
        """获取支持的求解器列表"""
        response = requests.get(f"{self.base_url}/solvers", timeout=self.timeout)
        return response.json().get("solvers", [])
    
    def solve(
        self, 
        problem_str: str, 
        puzzle_type: str,
        params: Optional[Dict] = None
    ) -> Dict[str, Any]:
        """
        求解谜题
        
        Args:
            problem_str: 谜题输入字符串
            puzzle_type: 谜题类型 (如 "sudoku", "akari", "slitherlink")
            params: 额外参数
            
        Returns:
            包含 status, grid, cpu_time 等的字典
        """
        response = requests.post(
            f"{self.base_url}/solve",
            json={
                "problem_str": problem_str,
                "puzzle_type": puzzle_type,
                "params": params or {}
            },
            timeout=self.timeout
        )
        
        if response.status_code != 200:
            raise Exception(f"API Error: {response.json().get('detail', response.text)}")
        
        return response.json()
    
    def solve_sudoku(self, grid: List[List[str]]) -> List[List[str]]:
        """便捷方法：求解数独"""
        rows = len(grid)
        cols = len(grid[0])
        problem_str = f"{rows} {cols}\n"
        problem_str += "\n".join(" ".join(row) for row in grid)
        
        result = self.solve(problem_str, "sudoku")
        return result["grid"]
    
    def print_grid(self, grid: List[List[Any]]):
        """格式化打印网格"""
        for row in grid:
            print(" ".join(f"{str(c):>2}" for c in row))


# 使用示例
if __name__ == "__main__":
    client = PuzzleKitClient()
    
    # 1. 健康检查
    print("=== Health Check ===")
    print(client.health_check())
    
    # 2. 求解数独
    print("\n=== Sudoku ===")
    result = client.solve(
        problem_str="""9 9
5 3 - - 7 - - - -
6 - - 1 9 5 - - -
- 9 8 - - - - 6 -
8 - - - 6 - - - 3
4 - - 8 - 3 - - 1
7 - - - 2 - - - 6
- 6 - - - - 2 8 -
- - - 4 1 9 - - 5
- - - - 8 - - 7 9""",
        puzzle_type="sudoku"
    )
    print(f"Status: {result['status']}")
    print(f"CPU Time: {result.get('cpu_time', 'N/A')}s")
    client.print_grid(result['grid'])
    
    # 3. 求解 Slitherlink
    print("\n=== Slitherlink ===")
    result = client.solve(
        problem_str="""10 18\n3 1 - - 1 - 3 - - 2 - 1 - 2 3 1 2 2\n- - 3 2 2 2 - - 1 - - - - 2 - 2 3 3\n2 2 2 2 - 2 2 - 1 - 2 - 0 2 - 1 - 2\n2 2 - 2 - 1 - - 1 - 0 1 2 - 3 2 - -\n3 1 - - - 2 - - 3 - - - 1 - - - - -\n- - - - - 1 - - - 3 - - 1 - - - 1 1\n- - 2 2 - 2 1 2 - 2 - - 2 - 1 - 2 3\n3 - 3 - 1 1 - 1 - 1 - 1 2 - 2 2 2 2\n2 2 2 - 2 - - - - 2 - - 2 1 1 1 - -\n0 2 3 1 2 - 3 - 1 - - 1 - 3 - - 3 2""",
        puzzle_type="slitherlink"
    )
    print(f"Status: {result['status']}")
    print(f"CPU Time: {result.get('cpu_time', 'N/A')}s")
    client.print_grid(result['grid'])

=== Health Check ===
{'status': 'running', 'version': '0.3.0'}

=== Sudoku ===
Status: Optimal
CPU Time: 0.00338411s
 5  3  4  6  7  8  9  1  2
 6  7  2  1  9  5  3  4  8
 1  9  8  3  4  2  5  6  7
 8  5  9  7  6  1  4  2  3
 4  2  6  8  5  3  7  9  1
 7  1  3  9  2  4  8  5  6
 9  6  1  5  3  7  2  8  4
 2  8  7  4  1  9  6  3  5
 3  4  5  2  8  6  1  7  9

=== Slitherlink ===
Status: Optimal
CPU Time: 1.089967056s
13  4  2  2  1 13  7 12  9  5 14  8  9  5 14  8 10  9
 5  5 14  9  5  6  8  -  1  4 11  4  1  6 11  5 13  7
 5  6  9  5  4  9  6  -  1  7 12  -  - 10 10  1  4 10
 6  9  7  5  6  2 11  4  2  8  -  2  3 12 11  5  7 13
11  4  8  2 10 10 10  1 13  6  1 14  8  1 14  -  8  1
14  -  1 12 10  8  9  7  4 11  4  9  4  -  9  6  2  1
11  4  3  5 13  6  2 10  3 12  -  1  6  -  2 10  9  7
14  1 14  1  4  8  8  8 10  2  -  2  9  7 12  9  6 10
 9  6  9  5  6  -  2  1 12  9  5 13  6  8  2  2  8 11
 -  9  7  4  9  7 13  7  4  1  7  4  9  7 12  9  7 12


In [3]:
import numpy as np
from puzzle_solver import nurikabe_solver as solver
# board = np.array([
    
# ])

board = "10 10\n- - 3 - - - - 2 - -\n- - - 3 - - - - - 2\n- - 2 - - - - - - -\n- - - 3 - - - - - 3\n- 3 - - - - - - - -\n- - - - 3 - - - - 3\n- - 2 - - - 2 - - -\n- - - - - 3 - - - -\n- - - - - - - - - -\n- 3 - - - - - 2 - 2".split("\n")[1:]
board = list(map(lambda x: x.split(" "), board))
board = list(map(lambda x: list(map(lambda y: " " if y == "-" else y, x)), board))
board = np.array(board)
# print(board)
binst = solver.Board(board=board)
solutions = binst.solve_and_print()


Solution found
    0   0   0   0   0   0   0   0   0   0  
    0   1   2   3   4   5   6   7   8   9  
  ┌───┬───┬───┬───┬───┬───┬───┬───┬───┬───┐
 0│   │   │ 3 │▒▒▒│▒▒▒│▒▒▒│▒▒▒│ 2 │▒▒▒│   │
  ├───┼───┼───┼───┼───┼───┼───┼───┼───┼───┤
 1│▒▒▒│▒▒▒│▒▒▒│ 3 │   │   │▒▒▒│   │▒▒▒│ 2 │
  ├───┼───┼───┼───┼───┼───┼───┼───┼───┼───┤
 2│▒▒▒│   │ 2 │▒▒▒│▒▒▒│▒▒▒│▒▒▒│▒▒▒│▒▒▒│▒▒▒│
  ├───┼───┼───┼───┼───┼───┼───┼───┼───┼───┤
 3│▒▒▒│▒▒▒│▒▒▒│ 3 │   │   │▒▒▒│   │   │ 3 │
  ├───┼───┼───┼───┼───┼───┼───┼───┼───┼───┤
 4│   │ 3 │▒▒▒│▒▒▒│▒▒▒│▒▒▒│▒▒▒│▒▒▒│▒▒▒│▒▒▒│
  ├───┼───┼───┼───┼───┼───┼───┼───┼───┼───┤
 5│   │▒▒▒│▒▒▒│   │ 3 │▒▒▒│   │▒▒▒│   │ 3 │
  ├───┼───┼───┼───┼───┼───┼───┼───┼───┼───┤
 6│▒▒▒│   │ 2 │▒▒▒│   │▒▒▒│ 2 │▒▒▒│   │▒▒▒│
  ├───┼───┼───┼───┼───┼───┼───┼───┼───┼───┤
 7│▒▒▒│▒▒▒│▒▒▒│▒▒▒│▒▒▒│ 3 │▒▒▒│▒▒▒│▒▒▒│▒▒▒│
  ├───┼───┼───┼───┼───┼───┼───┼───┼───┼───┤
 8│▒▒▒│   │   │▒▒▒│   │   │▒▒▒│   │▒▒▒│   │
  ├───┼───┼───┼───┼───┼───┼───┼───┼───┼───┤
 9│▒▒▒│ 3 │▒▒▒│▒▒▒│▒▒▒│▒▒▒│▒▒▒│ 2 │▒▒▒│ 2 │
  └───┴───┴───┴──

In [4]:
len("8225c438591c4823ca7256c4d95625ac768ee4e4a1a4aded9dbc717c1141dba864425ec16dc6352aadbd95a3ed411da98d97854dabd34d8dedaa6a1a277d4a979a1e292225552c18293765ae85c7ab73191b69ed77d76d39148c2b263a317e83b723d7a797a395c15cadbc5ad13b585b31b7bcabd93ab5a5249d55325844a421651a979ac4ddb7e97a5e771aa1d9cdb31b535946abe181a2172b4ab51b84b1d31d3d6593e9ea895154e54c17739cd323465614d51ea9789d1777571e96da8a55646123c58cc81281")

400